# 04 Model Development

本 Notebook 依據 Jain et al. (2024) *Predicting hospital length of stay using machine learning on a large open health dataset* 之 Methods，進行模型開發流程重現（Reproduction）。

本章僅執行原論文模型流程，不加入額外 Reanalysis。Feature Importance、SHAP、Error Analysis、Additional Visualizations 將於後續章節另行處理。

## 4.1 載入套件與設定路徑

原論文使用 Python、Pandas、Scikit-learn、PyTorch 與 CatBoost。本課程專案規範為全程使用 R 與 Jupyter Notebook，因此本章以 R 套件重現相同資料流與建模邏輯。

需要注意：原論文未使用 XGBoost 或 LightGBM，因此 Reproduction 階段不納入 XGBoost 與 LightGBM，以避免偏離原論文 Methods。

In [8]:
#--------------------------------------------------
# 4.1.0 套件安裝檢查（必要時才安裝）
#--------------------------------------------------

required_packages <- c(
  "data.table", "dplyr", "tidyr", "stringr", "forcats",
  "ggplot2", "caret", "Matrix", "ranger", "nnet",
  "glmnet", "tictoc"
)

missing_packages <- required_packages[
  !sapply(required_packages, requireNamespace, quietly = TRUE)
]

if (length(missing_packages) > 0) {
  install.packages(missing_packages)
}

# catboost 不在 CRAN stable binary 中，Apple Silicon / R 4.6.1 可能無法直接安裝。
# 因此本 Notebook 僅偵測是否已安裝；若未安裝，CatBoost 區塊會標示 Not run。


Installing packages into ‘/Users/pingl_macstudio_ai-lab/Library/R/arm64/4.6/library’
(as ‘lib’ is unspecified)

Warning message:
“package ‘catboost’ is not available for this version of R

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”



The downloaded binary packages are in
	/var/folders/33/nf67hss12z11tjl5fflfttfr0000gp/T//RtmpinQ4e0/downloaded_packages


In [9]:
#--------------------------------------------------
# 4.1 載入套件與設定路徑
#--------------------------------------------------

suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(forcats)
  library(ggplot2)
  library(caret)
  library(Matrix)
  library(ranger)
  library(nnet)
})

# CatBoost 為原論文模型之一；若尚未安裝，相關區塊會自動略過並保留說明。
has_catboost <- requireNamespace("catboost", quietly = TRUE)
if (has_catboost) {
  library(catboost)
} else {
  message("catboost 套件尚未安裝：CatBoost 模型區塊將不執行。")
}

set.seed(2024)

# 若 Notebook 放在 NYCU_R_FinalProject/notebook/，project_path 設為上一層。
project_path   <- normalizePath("..", mustWork = FALSE)
processed_path <- file.path(project_path, "data", "processed")
output_path    <- file.path(project_path, "output")
figure_path    <- file.path(project_path, "figures")
model_path     <- file.path(output_path, "models")

# 若在其他位置測試，可自動使用目前工作目錄的上層或目前目錄。
if (!dir.exists(processed_path)) {
  project_path   <- normalizePath(".", mustWork = FALSE)
  processed_path <- file.path(project_path, "data", "processed")
  output_path    <- file.path(project_path, "output")
  figure_path    <- file.path(project_path, "figures")
  model_path     <- file.path(output_path, "models")
}

dir.create(output_path, recursive = TRUE, showWarnings = FALSE)
dir.create(figure_path, recursive = TRUE, showWarnings = FALSE)
dir.create(model_path, recursive = TRUE, showWarnings = FALSE)

list(
  project_path = project_path,
  processed_path = processed_path,
  output_path = output_path,
  figure_path = figure_path,
  model_path = model_path,
  has_catboost = has_catboost
)

catboost 套件尚未安裝：CatBoost 模型區塊將不執行。



$project_path
[1] "/Users/pingl_macstudio_ai-lab/Desktop/NYCU_R_FinalProject"

$processed_path
[1] "/Users/pingl_macstudio_ai-lab/Desktop/NYCU_R_FinalProject/data/processed"

$output_path
[1] "/Users/pingl_macstudio_ai-lab/Desktop/NYCU_R_FinalProject/output"

$figure_path
[1] "/Users/pingl_macstudio_ai-lab/Desktop/NYCU_R_FinalProject/figures"

$model_path
[1] "/Users/pingl_macstudio_ai-lab/Desktop/NYCU_R_FinalProject/output/models"

$has_catboost
[1] FALSE

## 4.2 讀取前處理後資料

依據 02_Preprocessing.ipynb 的輸出，本章讀取 newborn 與 non-newborn 兩個資料分區。原論文指出約 10% 資料為 newborn，並針對 newborn 與 non-newborn 建立不同模型；其中 Birth Weight 僅用於 newborn，不納入 non-newborn 模型。

In [10]:
#--------------------------------------------------
# 4.2 讀取前處理後資料
#--------------------------------------------------

newborn_file     <- file.path(processed_path, "df_newborn.rds")
non_newborn_file <- file.path(processed_path, "df_non_newborn.rds")

stopifnot(file.exists(newborn_file))
stopifnot(file.exists(non_newborn_file))

df_newborn <- readRDS(newborn_file)
df_non_newborn <- readRDS(non_newborn_file)

model_data_summary <- tibble(
  Dataset = c("Newborn", "Non-newborn"),
  Rows = c(nrow(df_newborn), nrow(df_non_newborn)),
  Columns = c(ncol(df_newborn), ncol(df_non_newborn))
)

model_data_summary

Dataset,Rows,Columns
<chr>,<int>,<int>
Newborn,229290,25
Non-newborn,2108814,24


## 4.3 建立 Target Variable

原論文以住院天數（Length of Stay, LoS）作為預測目標。Regression models 使用 LoS 連續值，範圍為 1 至 120 天。Classification models 則將 LoS 離散化為五個類別：1 day、2 days、3 days、4–6 days、>6 days。

In [11]:
#--------------------------------------------------
# 4.3 建立 Target Variable
#--------------------------------------------------

prepare_target <- function(df) {
  df %>%
    mutate(
      LOS = as.numeric(`Length of Stay`),
      LOS_class = case_when(
        LOS == 1 ~ "1",
        LOS == 2 ~ "2",
        LOS == 3 ~ "3",
        LOS >= 4 & LOS <= 6 ~ "4-6",
        LOS > 6 ~ ">6",
        TRUE ~ NA_character_
      ),
      LOS_class = factor(LOS_class, levels = c("1", "2", "3", "4-6", ">6"))
    ) %>%
    filter(!is.na(LOS), !is.na(LOS_class), LOS >= 1, LOS <= 120)
}

df_newborn_model <- prepare_target(df_newborn)
df_non_newborn_model <- prepare_target(df_non_newborn)

target_summary <- bind_rows(
  df_newborn_model %>% summarise(Dataset = "Newborn", Mean = mean(LOS), SD = sd(LOS), Min = min(LOS), Q1 = quantile(LOS, .25), Median = median(LOS), Q3 = quantile(LOS, .75), Max = max(LOS)),
  df_non_newborn_model %>% summarise(Dataset = "Non-newborn", Mean = mean(LOS), SD = sd(LOS), Min = min(LOS), Q1 = quantile(LOS, .25), Median = median(LOS), Q3 = quantile(LOS, .75), Max = max(LOS))
)

target_summary

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `LOS = as.numeric(`Length of Stay`)`.
Caused by warning:
! NAs introduced by coercion”
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `LOS = as.numeric(`Length of Stay`)`.
Caused by warning:
! NAs introduced by coercion”


Dataset,Mean,SD,Min,Q1,Median,Q3,Max
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Newborn,3.711209,7.493480,1,2,2,3,119
Non-newborn,5.503642,7.287771,1,2,3,6,119


## 4.4 建立 90% Training / 10% Holdout Test Split

原論文使用 10% 資料作為 holdout test set；剩餘 90% 資料用於 tenfold cross-validation，以訓練模型並決定參數。本節依照該設定建立 newborn 與 non-newborn 的資料切分。

In [12]:
#--------------------------------------------------
# 4.4 建立 90% Training / 10% Holdout Test Split
#--------------------------------------------------

split_train_test <- function(df, p_train = 0.90, seed = 2024) {
  set.seed(seed)
  idx <- caret::createDataPartition(df$LOS_class, p = p_train, list = FALSE)
  list(
    train = df[idx, ],
    test  = df[-idx, ]
  )
}

split_newborn <- split_train_test(df_newborn_model)
split_non_newborn <- split_train_test(df_non_newborn_model)

newborn_train <- split_newborn$train
newborn_test  <- split_newborn$test
non_newborn_train <- split_non_newborn$train
non_newborn_test  <- split_non_newborn$test

split_summary <- tibble(
  Dataset = c("Newborn training", "Newborn holdout test", "Non-newborn training", "Non-newborn holdout test"),
  Rows = c(nrow(newborn_train), nrow(newborn_test), nrow(non_newborn_train), nrow(non_newborn_test)),
  Percentage = c(
    nrow(newborn_train) / nrow(df_newborn_model),
    nrow(newborn_test) / nrow(df_newborn_model),
    nrow(non_newborn_train) / nrow(df_non_newborn_model),
    nrow(non_newborn_test) / nrow(df_non_newborn_model)
  ) * 100
) %>%
  mutate(Percentage = round(Percentage, 2))

split_summary

Dataset,Rows,Percentage
<chr>,<int>,<dbl>
Newborn training,206146,90
Newborn holdout test,22902,10
Non-newborn training,1896579,90
Non-newborn holdout test,210728,10


## 4.5 Feature Encoding Functions

原論文比較三種 encoding：one-hot encoding、target encoding，以及 one-hot + target encoding。Target encoding 的方式為以類別對應的 LoS mean 與 LoS median 形成分布依賴特徵。本 Notebook 以訓練集建立 encoding mapping，再套用到 holdout test set，以避免資料洩漏。

In [13]:
#--------------------------------------------------
# 4.5 Feature Encoding Functions
#--------------------------------------------------

get_feature_columns <- function(df) {
  setdiff(names(df), c("Length of Stay", "LOS", "LOS_class"))
}

convert_features_to_factor <- function(df, feature_cols) {
  df %>%
    mutate(across(all_of(feature_cols), ~ as.factor(as.character(.x))))
}

build_target_encoding_map <- function(train_df, feature_cols, target_col = "LOS") {
  global_mean <- mean(train_df[[target_col]], na.rm = TRUE)
  global_median <- median(train_df[[target_col]], na.rm = TRUE)
  global_value <- global_mean * global_median

  maps <- lapply(feature_cols, function(col) {
    train_df %>%
      group_by(.data[[col]]) %>%
      summarise(
        te_mean = mean(.data[[target_col]], na.rm = TRUE),
        te_median = median(.data[[target_col]], na.rm = TRUE),
        te_value = te_mean * te_median,
        .groups = "drop"
      ) %>%
      rename(level = 1) %>%
      mutate(level = as.character(level))
  })
  names(maps) <- feature_cols

  list(maps = maps, global_value = global_value)
}

apply_target_encoding <- function(df, feature_cols, encoding_object) {
  encoded <- df %>% select(all_of(feature_cols))

  for (col in feature_cols) {
    map_tbl <- encoding_object$maps[[col]]
    tmp <- tibble(level = as.character(df[[col]])) %>%
      left_join(map_tbl, by = "level") %>%
      mutate(te_value = if_else(is.na(te_value), encoding_object$global_value, te_value))
    encoded[[paste0(col, "__te")]] <- tmp$te_value
  }

  encoded %>%
    select(ends_with("__te")) %>%
    as.data.frame()
}


r2_score <- function(actual, predicted) {
  1 - sum((actual - predicted)^2, na.rm = TRUE) / sum((actual - mean(actual, na.rm = TRUE))^2, na.rm = TRUE)
}

rmse_score <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2, na.rm = TRUE))
}

mae_score <- function(actual, predicted) {
  mean(abs(actual - predicted), na.rm = TRUE)
}

regression_metrics <- function(actual, predicted, model_name, dataset_name) {
  test_cor <- suppressWarnings(cor.test(actual, predicted))
  tibble(
    Dataset = dataset_name,
    Model = model_name,
    R2 = r2_score(actual, predicted),
    RMSE = rmse_score(actual, predicted),
    MAE = mae_score(actual, predicted),
    P_value = test_cor$p.value
  )
}

## 4.6 Regression Models

原論文 regression models 包含 Linear Regression、CatBoost Regression 與 Random Forest Regression。Random Forest Regression 使用 target encoding；Linear Regression 比較 one-hot encoding 與 one-hot + target encoding；CatBoost Regression 用於模型效能與 minimal feature sets 探索。

為避免首次執行耗時過長，以下提供 `USE_SUBSAMPLE` 開關。正式重現請維持 `FALSE`；若僅測試流程，可改為 `TRUE`。

In [14]:
#--------------------------------------------------
# 4.6 Regression Models：資料準備
#--------------------------------------------------

USE_SUBSAMPLE <- FALSE
SUBSAMPLE_N_NON_NEWBORN <- 200000
SUBSAMPLE_N_NEWBORN <- 100000

make_model_sample <- function(train_df, test_df, n_train, seed = 2024) {
  if (!USE_SUBSAMPLE) {
    return(list(train = train_df, test = test_df))
  }
  set.seed(seed)
  train_sample <- train_df %>% slice_sample(n = min(n_train, nrow(train_df)))
  list(train = train_sample, test = test_df)
}

nb_reg_data <- make_model_sample(newborn_train, newborn_test, SUBSAMPLE_N_NEWBORN)
nnb_reg_data <- make_model_sample(non_newborn_train, non_newborn_test, SUBSAMPLE_N_NON_NEWBORN)

newborn_feature_cols <- get_feature_columns(nb_reg_data$train)
non_newborn_feature_cols <- get_feature_columns(nnb_reg_data$train)

nb_train_x <- convert_features_to_factor(nb_reg_data$train, newborn_feature_cols)
nb_test_x  <- convert_features_to_factor(nb_reg_data$test, newborn_feature_cols)
nnb_train_x <- convert_features_to_factor(nnb_reg_data$train, non_newborn_feature_cols)
nnb_test_x  <- convert_features_to_factor(nnb_reg_data$test, non_newborn_feature_cols)

list(
  newborn_train_rows = nrow(nb_train_x),
  newborn_test_rows = nrow(nb_test_x),
  non_newborn_train_rows = nrow(nnb_train_x),
  non_newborn_test_rows = nrow(nnb_test_x)
)

$newborn_train_rows
[1] 206146

$newborn_test_rows
[1] 22902

$non_newborn_train_rows
[1] 1896579

$non_newborn_test_rows
[1] 210728

In [15]:
#--------------------------------------------------
# 4.6.1 Linear Regression with One-Hot Encoding
#--------------------------------------------------

build_onehot_matrix <- function(train_df, test_df, feature_cols) {
  
  train_x <- train_df %>%
    select(all_of(feature_cols)) %>%
    mutate(.dataset = "train")
  
  test_x <- test_df %>%
    select(all_of(feature_cols)) %>%
    mutate(.dataset = "test")
  
  all_x <- bind_rows(train_x, test_x)
  
  all_x <- all_x %>%
    mutate(across(everything(), ~ as.character(.x))) %>%
    mutate(across(everything(), ~ ifelse(is.na(.x), "Missing", .x)))
  
  dataset_flag <- all_x$.dataset
  
  model_x <- all_x %>%
    select(-.dataset)
  
  # 修正欄位名稱，避免 sparse.model.matrix 公式解析錯誤
  original_names <- names(model_x)
  safe_names <- make.names(original_names, unique = TRUE)
  names(model_x) <- safe_names
  
  valid_cols <- names(model_x)[
    sapply(model_x, function(x) length(unique(x)) >= 2)
  ]
  
  removed_cols <- setdiff(names(model_x), valid_cols)
  
  model_x_valid <- model_x %>%
    select(all_of(valid_cols))
  
  # 全部轉為 factor，並移除未使用 levels
  model_x_valid <- model_x_valid %>%
    mutate(across(everything(), ~ droplevels(as.factor(.x))))
  
  x_mat <- Matrix::sparse.model.matrix(
    object = ~ . - 1,
    data = model_x_valid
  )
  
  train_mat <- x_mat[dataset_flag == "train", , drop = FALSE]
  test_mat  <- x_mat[dataset_flag == "test", , drop = FALSE]
  
  return(list(
    train = train_mat,
    test = test_mat,
    columns = colnames(x_mat),
    removed_single_level_cols = removed_cols,
    original_feature_names = original_names,
    safe_feature_names = safe_names
  ))
}


run_linear_onehot <- function(train_df, test_df, feature_cols, dataset_name) {
  
  message("Building one-hot matrix for ", dataset_name, " ...")
  
  oh <- build_onehot_matrix(
    train_df = train_df,
    test_df = test_df,
    feature_cols = feature_cols
  )
  
  message("One-hot columns: ", ncol(oh$train))
  
  if (length(oh$removed_single_level_cols) > 0) {
    message(
      "Removed single-level columns: ",
      paste(oh$removed_single_level_cols, collapse = ", ")
    )
  }
  
  if (!requireNamespace("glmnet", quietly = TRUE)) {
    stop("請先安裝 glmnet 套件以執行 large sparse linear regression。")
  }
  
  # glmnet 可處理 sparse matrix；alpha = 0 為 ridge regression
  fit <- glmnet::cv.glmnet(
    x = oh$train,
    y = train_df$LOS,
    alpha = 0,
    family = "gaussian",
    nfolds = 10,
    standardize = FALSE
  )
  
  pred <- as.numeric(
    predict(fit, newx = oh$test, s = "lambda.min")
  )
  
  metrics <- regression_metrics(
    actual = test_df$LOS,
    predicted = pred,
    model_name = "Linear Regression - One Hot",
    dataset_name = dataset_name
  )
  
  return(list(
    model = fit,
    prediction = pred,
    metrics = metrics,
    onehot_info = list(
      columns = oh$columns,
      removed_single_level_cols = oh$removed_single_level_cols
    )
  ))
}


linear_onehot_results <- list()

# Newborn
linear_onehot_results$newborn <- run_linear_onehot(
  train_df = nb_train_x,
  test_df = nb_test_x,
  feature_cols = newborn_feature_cols,
  dataset_name = "Newborn"
)

# Non-newborn
# 注意：完整 non-newborn one-hot matrix 可能非常大，正式執行需較長時間與較大記憶體。
linear_onehot_results$non_newborn <- run_linear_onehot(
  train_df = nnb_train_x,
  test_df = nnb_test_x,
  feature_cols = non_newborn_feature_cols,
  dataset_name = "Non-newborn"
)

linear_onehot_table <- bind_rows(
  linear_onehot_results$newborn$metrics,
  linear_onehot_results$non_newborn$metrics
)

linear_onehot_table

Building one-hot matrix for Newborn ...

One-hot columns: 851

Removed single-level columns: Age.Group, APR.MDC.Code

Building one-hot matrix for Non-newborn ...

One-hot columns: 1576



Dataset,Model,R2,RMSE,MAE,P_value
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Newborn,Linear Regression - One Hot,0.5307364,4.919452,1.705203,0
Non-newborn,Linear Regression - One Hot,0.3235852,6.009947,3.032439,0


In [16]:
#--------------------------------------------------
# 4.6.2 Random Forest Regression with Target Encoding
#--------------------------------------------------

run_rf_target_regression <- function(train_df, test_df, feature_cols, dataset_name) {
  
  message("Running Random Forest Regression with Target Encoding: ", dataset_name)
  
  te_obj <- build_target_encoding_map(train_df, feature_cols)
  
  train_te <- apply_target_encoding(train_df, feature_cols, te_obj)
  test_te  <- apply_target_encoding(test_df, feature_cols, te_obj)
  
  # 修正欄位名稱，避免 ranger formula interface 錯誤
  original_names <- names(train_te)
  safe_names <- make.names(original_names, unique = TRUE)
  names(train_te) <- safe_names
  names(test_te)  <- safe_names
  
  # 加入目標變項
  train_te$LOS <- train_df$LOS
  
  # 補 NA：test 出現 train 沒看過的類別時可能產生 NA
  for (col in setdiff(names(train_te), "LOS")) {
    med_value <- median(train_te[[col]], na.rm = TRUE)
    
    if (is.na(med_value)) {
      med_value <- 0
    }
    
    train_te[[col]][is.na(train_te[[col]])] <- med_value
    test_te[[col]][is.na(test_te[[col]])]   <- med_value
  }
  
  fit <- ranger::ranger(
    formula = LOS ~ .,
    data = train_te,
    num.trees = 10,
    max.depth = 10,
    seed = 2024,
    importance = "impurity"
  )
  
  pred <- predict(
    fit,
    data = test_te
  )$predictions
  
  metrics <- regression_metrics(
    actual = test_df$LOS,
    predicted = pred,
    model_name = "Random Forest Regression - Target",
    dataset_name = dataset_name
  )
  
  return(list(
    model = fit,
    prediction = pred,
    metrics = metrics,
    target_encoding = te_obj,
    original_feature_names = original_names,
    safe_feature_names = safe_names
  ))
}


rf_reg_results <- list()

rf_reg_results$newborn <- run_rf_target_regression(
  train_df = nb_train_x,
  test_df = nb_test_x,
  feature_cols = newborn_feature_cols,
  dataset_name = "Newborn"
)

rf_reg_results$non_newborn <- run_rf_target_regression(
  train_df = nnb_train_x,
  test_df = nnb_test_x,
  feature_cols = non_newborn_feature_cols,
  dataset_name = "Non-newborn"
)

rf_regression_table <- bind_rows(
  rf_reg_results$newborn$metrics,
  rf_reg_results$non_newborn$metrics
)

rf_regression_table

Running Random Forest Regression with Target Encoding: Newborn

Running Random Forest Regression with Target Encoding: Non-newborn



Dataset,Model,R2,RMSE,MAE,P_value
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Newborn,Random Forest Regression - Target,0.7453964,3.623603,1.253503,0
Non-newborn,Random Forest Regression - Target,0.3962736,5.677854,2.820428,0


In [17]:
#--------------------------------------------------
# 4.6.3 CatBoost Regression
#--------------------------------------------------

run_catboost_regression <- function(train_df,
                                    test_df,
                                    feature_cols,
                                    dataset_name) {
  
  if (!exists("has_catboost") || !isTRUE(has_catboost)) {
    
    return(
      tibble(
        Dataset = dataset_name,
        Model = "CatBoost Regression",
        Status = "Not run: catboost package unavailable in current R environment",
        R2 = NA_real_,
        RMSE = NA_real_,
        MAE = NA_real_,
        P_value = NA_real_
      )
    )
  }
  
  stop("CatBoost package detected, but R implementation block has not been activated.")
}


catboost_regression_table <- bind_rows(
  run_catboost_regression(
    train_df = nb_train_x,
    test_df = nb_test_x,
    feature_cols = newborn_feature_cols,
    dataset_name = "Newborn"
  ),
  run_catboost_regression(
    train_df = nnb_train_x,
    test_df = nnb_test_x,
    feature_cols = non_newborn_feature_cols,
    dataset_name = "Non-newborn"
  )
)

catboost_regression_table

Dataset,Model,Status,R2,RMSE,MAE,P_value
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Newborn,CatBoost Regression,Not run: catboost package unavailable in current R environment,NA,NA,NA,NA
Non-newborn,CatBoost Regression,Not run: catboost package unavailable in current R environment,NA,NA,NA,NA


In [19]:
#--------------------------------------------------
# 4.6.4 Regression Model Comparison
#--------------------------------------------------

regression_result_list <- list(
  linear_onehot_results$newborn$metrics,
  linear_onehot_results$non_newborn$metrics,
  rf_reg_results$newborn$metrics,
  rf_reg_results$non_newborn$metrics
)

# 若 CatBoost 有成功建立，才加入比較表
if (exists("cat_reg_results")) {
  regression_result_list <- c(
    regression_result_list,
    list(
      cat_reg_results$newborn$metrics,
      cat_reg_results$non_newborn$metrics
    )
  )
}

# 若使用的是 catboost_regression_table，則加入 Not run 表格
if (exists("catboost_regression_table")) {
  regression_result_list <- c(
    regression_result_list,
    list(catboost_regression_table)
  )
}

regression_comparison_table <- bind_rows(regression_result_list) %>%
  mutate(across(where(is.numeric), ~ round(.x, 4)))

regression_comparison_table

Dataset,Model,R2,RMSE,MAE,P_value,Status
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Newborn,Linear Regression - One Hot,0.5307,4.9195,1.7052,0,NA
Non-newborn,Linear Regression - One Hot,0.3236,6.0099,3.0324,0,NA
Newborn,Random Forest Regression - Target,0.7454,3.6236,1.2535,0,NA
Non-newborn,Random Forest Regression - Target,0.3963,5.6779,2.8204,0,NA
Newborn,CatBoost Regression,NA,NA,NA,NA,Not run: catboost package unavailable in current R environment
Non-newborn,CatBoost Regression,NA,NA,NA,NA,Not run: catboost package unavailable in current R environment


## 4.7 Classification Models

原論文將 LoS 分為五類：1、2、3、4–6、>6 days，並建立 Multinomial Logistic Regression、Random Forest Classifier 與 CatBoost Classifier。本節依照該流程進行分類模型重現。

In [20]:
#--------------------------------------------------
# 4.7 Classification Models：評估函數
#--------------------------------------------------

classification_metrics <- function(actual, predicted, model_name, dataset_name) {
  actual <- factor(actual, levels = c("1", "2", "3", "4-6", ">6"))
  predicted <- factor(predicted, levels = c("1", "2", "3", "4-6", ">6"))
  cm <- caret::confusionMatrix(predicted, actual)

  by_class <- as.data.frame(cm$byClass)
  by_class$Class <- rownames(by_class)
  rownames(by_class) <- NULL

  macro_f1 <- mean(by_class$F1, na.rm = TRUE)

  tibble(
    Dataset = dataset_name,
    Model = model_name,
    Accuracy = as.numeric(cm$overall["Accuracy"]),
    Macro_F1 = macro_f1
  )
}

make_confusion_table <- function(actual, predicted) {
  table(
    Actual = factor(actual, levels = c("1", "2", "3", "4-6", ">6")),
    Predicted = factor(predicted, levels = c("1", "2", "3", "4-6", ">6"))
  )
}

In [26]:
#--------------------------------------------------
# 4.7.1 Multinomial Logistic Regression with One-Hot Encoding
#--------------------------------------------------

run_multinom_logistic <- function(train_df, test_df, feature_cols, dataset_name,
                                  max_train_n = 100000,
                                  max_test_n = 30000) {
  
  set.seed(2024)
  
  # 為避免 one-hot dense matrix 記憶體過大，分類模型先依論文流程抽樣重現
  if (nrow(train_df) > max_train_n) {
    train_df <- train_df %>% sample_n(max_train_n)
  }
  
  if (nrow(test_df) > max_test_n) {
    test_df <- test_df %>% sample_n(max_test_n)
  }
  
  oh <- build_onehot_matrix(train_df, test_df, feature_cols)
  
  train_dense <- as.data.frame(as.matrix(oh$train))
  test_dense  <- as.data.frame(as.matrix(oh$test))
  
  train_dense$LOS_class <- train_df$LOS_class
  
  fit <- nnet::multinom(
    LOS_class ~ .,
    data = train_dense,
    MaxNWts = 1000000,
    maxit = 100,
    trace = TRUE
  )
  
  pred <- predict(fit, newdata = test_dense)
  
  list(
    model = fit,
    prediction = pred,
    metrics = classification_metrics(
      test_df$LOS_class,
      pred,
      "Multinomial Logistic Regression",
      dataset_name
    ),
    confusion_matrix = make_confusion_table(test_df$LOS_class, pred),
    sampled_train_n = nrow(train_df),
    sampled_test_n = nrow(test_df)
  )
}

In [25]:
multinom_results <- list()

multinom_results$newborn <- run_multinom_logistic(
  nb_train_x, nb_test_x, newborn_feature_cols, "Newborn",
  max_train_n = 80000,
  max_test_n = 20000
)

multinom_results$non_newborn <- run_multinom_logistic(
  nnb_train_x, nnb_test_x, non_newborn_feature_cols, "Non-newborn",
  max_train_n = 100000,
  max_test_n = 30000
)

bind_rows(
  multinom_results$newborn$metrics,
  multinom_results$non_newborn$metrics
)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.2 GiB”


由於 non-newborn 資料筆數超過 210 萬，one-hot encoding 後轉為 dense matrix 會造成記憶體負擔。因此本節依原論文方法重現 multinomial logistic regression，但於本地 R 環境採固定亂數抽樣執行，以確保 Notebook 可重現且可於合理時間內完成。

In [29]:
#--------------------------------------------------
# 4.7.2 Random Forest Classifier with Target Encoding
#--------------------------------------------------

if (!requireNamespace("tictoc", quietly = TRUE)) {
  install.packages("tictoc")
}
library(tictoc)

run_rf_classifier <- function(train_df,
                              test_df,
                              feature_cols,
                              dataset_name,
                              max_train_n = 100000,
                              max_test_n = 30000) {
  
  message("======================================")
  message("Running Random Forest Classifier: ", dataset_name)
  message("Original train rows: ", format(nrow(train_df), big.mark = ","))
  message("Original test rows : ", format(nrow(test_df), big.mark = ","))
  
  set.seed(2024)
  
  tic("Sampling")
  
  if (nrow(train_df) > max_train_n) {
    train_df <- train_df %>% sample_n(max_train_n)
  }
  
  if (nrow(test_df) > max_test_n) {
    test_df <- test_df %>% sample_n(max_test_n)
  }
  
  toc()
  
  message("Sampled train rows: ", format(nrow(train_df), big.mark = ","))
  message("Sampled test rows : ", format(nrow(test_df), big.mark = ","))
  
  tic("Target Encoding")
  
  te_obj <- build_target_encoding_map(train_df, feature_cols)
  train_te <- apply_target_encoding(train_df, feature_cols, te_obj)
  test_te  <- apply_target_encoding(test_df, feature_cols, te_obj)
  
  toc()
  
  message("✓ Target encoding completed")
  
  tic("Fix column names and missing values")
  
  original_names <- names(train_te)
  safe_names <- make.names(original_names, unique = TRUE)
  names(train_te) <- safe_names
  names(test_te)  <- safe_names
  
  train_te$LOS_class <- as.factor(train_df$LOS_class)
  
  for (col in setdiff(names(train_te), "LOS_class")) {
    
    med_value <- median(train_te[[col]], na.rm = TRUE)
    
    if (is.na(med_value)) {
      med_value <- 0
    }
    
    train_te[[col]][is.na(train_te[[col]])] <- med_value
    test_te[[col]][is.na(test_te[[col]])]   <- med_value
  }
  
  toc()
  
  message("✓ Column names and missing values fixed")
  
  tic("Random Forest training")
  
  fit <- ranger::ranger(
    formula = LOS_class ~ .,
    data = train_te,
    num.trees = 10,
    max.depth = 10,
    probability = FALSE,
    seed = 2024,
    importance = "impurity",
    verbose = TRUE
  )
  
  toc()
  
  message("✓ Random Forest training completed")
  
  tic("Prediction")
  
  pred <- predict(
    fit,
    data = test_te
  )$predictions
  
  pred <- as.factor(pred)
  
  toc()
  
  message("✓ Prediction completed")
  
  tic("Evaluation")
  
  metrics <- classification_metrics(
    actual = as.factor(test_df$LOS_class),
    predicted = pred,
    model_name = "Random Forest Classifier - Target",
    dataset_name = dataset_name
  )
  
  confusion_matrix <- make_confusion_table(
    actual = as.factor(test_df$LOS_class),
    predicted = pred
  )
  
  toc()
  
  message("✓ Finished ", dataset_name)
  message("======================================")
  
  return(list(
    model = fit,
    actual = factor(test_df$LOS_class, levels = c("1", "2", "3", "4-6", ">6")),
    prediction = pred,
    metrics = metrics,
    confusion_matrix = confusion_matrix,
    target_encoding = te_obj,
    sampled_train_n = nrow(train_df),
    sampled_test_n = nrow(test_df),
    original_feature_names = original_names,
    safe_feature_names = safe_names
  ))
}


rf_cls_results <- list()

rf_cls_results$newborn <- run_rf_classifier(
  train_df = nb_train_x,
  test_df = nb_test_x,
  feature_cols = newborn_feature_cols,
  dataset_name = "Newborn",
  max_train_n = 80000,
  max_test_n = 20000
)

rf_cls_results$non_newborn <- run_rf_classifier(
  train_df = nnb_train_x,
  test_df = nnb_test_x,
  feature_cols = non_newborn_feature_cols,
  dataset_name = "Non-newborn",
  max_train_n = 100000,
  max_test_n = 30000
)

rf_classifier_table <- bind_rows(
  rf_cls_results$newborn$metrics,
  rf_cls_results$non_newborn$metrics
)

rf_classifier_table

Installing package into ‘/Users/pingl_macstudio_ai-lab/Library/R/arm64/4.6/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/33/nf67hss12z11tjl5fflfttfr0000gp/T//RtmpinQ4e0/downloaded_packages



Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift



Running Random Forest Classifier: Newborn

Original train rows: 206,146

Original test rows : 22,902



Sampling: 0.007 sec elapsed


Sampled train rows: 80,000

Sampled test rows : 20,000



...

Prediction: 0.041 sec elapsed


✓ Prediction completed



Evaluation: 0.006 sec elapsed


✓ Finished Non-newborn




Dataset,Model,Accuracy,Macro_F1
<chr>,<chr>,<dbl>,<dbl>
Newborn,Random Forest Classifier - Target,0.5922500,0.3867485
Non-newborn,Random Forest Classifier - Target,0.4530333,0.4218749


In [31]:
#--------------------------------------------------
# 4.7.3 CatBoost Classifier
#--------------------------------------------------

run_catboost_classifier <- function(train_df,
                                    test_df,
                                    feature_cols,
                                    dataset_name,
                                    max_train_n = 100000,
                                    max_test_n = 30000) {
  
  message("======================================")
  message("Running CatBoost Classifier: ", dataset_name)
  
  if (!exists("has_catboost") || !isTRUE(has_catboost)) {
    
    message("CatBoost package unavailable. Skip: ", dataset_name)
    
    return(list(
      model = NULL,
      actual = factor(test_df$LOS_class, levels = c("1", "2", "3", "4-6", ">6")),
      prediction = factor(
        rep(NA_character_, nrow(test_df)),
        levels = c("1", "2", "3", "4-6", ">6")
      ),
      metrics = tibble(
        Dataset = dataset_name,
        Model = "CatBoost Classifier",
        Status = "Not run: catboost package unavailable in current R environment",
        Accuracy = NA_real_,
        Macro_F1 = NA_real_
      ),
      confusion_matrix = NULL,
      sampled_train_n = NA_integer_,
      sampled_test_n = NA_integer_
    ))
  }
  
  set.seed(2024)
  
  if (nrow(train_df) > max_train_n) {
    train_df <- train_df %>% sample_n(max_train_n)
  }
  
  if (nrow(test_df) > max_test_n) {
    test_df <- test_df %>% sample_n(max_test_n)
  }
  
  message("Sampled train rows: ", format(nrow(train_df), big.mark = ","))
  message("Sampled test rows : ", format(nrow(test_df), big.mark = ","))
  
  train_x <- train_df %>%
    select(all_of(feature_cols)) %>%
    mutate(across(everything(), ~ as.character(.x))) %>%
    mutate(across(everything(), ~ ifelse(is.na(.x), "Missing", .x)))
  
  test_x <- test_df %>%
    select(all_of(feature_cols)) %>%
    mutate(across(everything(), ~ as.character(.x))) %>%
    mutate(across(everything(), ~ ifelse(is.na(.x), "Missing", .x)))
  
  cat_features <- seq_along(feature_cols) - 1
  
  y_train <- as.integer(as.factor(train_df$LOS_class)) - 1
  
  train_pool <- catboost.load_pool(
    data = train_x,
    label = y_train,
    cat_features = cat_features
  )
  
  test_pool <- catboost.load_pool(
    data = test_x,
    cat_features = cat_features
  )
  
  params <- list(
    loss_function = "MultiClass",
    eval_metric = "Accuracy",
    iterations = 300,
    depth = 6,
    learning_rate = 0.1,
    random_seed = 2024,
    verbose = 100
  )
  
  fit <- catboost.train(
    train_pool,
    params = params
  )
  
  pred_raw <- catboost.predict(
    fit,
    test_pool,
    prediction_type = "Class"
  )
  
  class_levels <- levels(as.factor(train_df$LOS_class))
  
  pred <- factor(
    class_levels[as.integer(pred_raw) + 1],
    levels = class_levels
  )
  
  metrics <- classification_metrics(
    actual = factor(test_df$LOS_class, levels = class_levels),
    predicted = pred,
    model_name = "CatBoost Classifier",
    dataset_name = dataset_name
  ) %>%
    mutate(Status = "Completed", .after = Model)
  
  confusion_matrix <- make_confusion_table(
    actual = factor(test_df$LOS_class, levels = class_levels),
    predicted = pred
  )
  
  message("✓ Finished CatBoost Classifier: ", dataset_name)
  message("======================================")
  
  return(list(
    model = fit,
    actual = factor(test_df$LOS_class, levels = class_levels),
    prediction = pred,
    probability = probability,
    metrics = metrics,
    confusion_matrix = confusion_matrix,
    sampled_train_n = nrow(train_df),
    sampled_test_n = nrow(test_df)
  ))
}


cat_cls_results <- list()

cat_cls_results$newborn <- run_catboost_classifier(
  train_df = nb_train_x,
  test_df = nb_test_x,
  feature_cols = newborn_feature_cols,
  dataset_name = "Newborn",
  max_train_n = 80000,
  max_test_n = 20000
)

cat_cls_results$non_newborn <- run_catboost_classifier(
  train_df = nnb_train_x,
  test_df = nnb_test_x,
  feature_cols = non_newborn_feature_cols,
  dataset_name = "Non-newborn",
  max_train_n = 100000,
  max_test_n = 30000
)

catboost_classifier_table <- bind_rows(
  cat_cls_results$newborn$metrics,
  cat_cls_results$non_newborn$metrics
)

catboost_classifier_table


Running CatBoost Classifier: Newborn

CatBoost package unavailable. Skip: Newborn


Running CatBoost Classifier: Non-newborn

CatBoost package unavailable. Skip: Non-newborn



Dataset,Model,Status,Accuracy,Macro_F1
<chr>,<chr>,<chr>,<dbl>,<dbl>
Newborn,CatBoost Classifier,Not run: catboost package unavailable in current R environment,NA,NA
Non-newborn,CatBoost Classifier,Not run: catboost package unavailable in current R environment,NA,NA


In [32]:
#--------------------------------------------------
# 4.7.4 Classification Model Summary
#--------------------------------------------------

classification_result_list <- list()

if (exists("multinom_results")) {
  classification_result_list <- c(
    classification_result_list,
    list(
      multinom_results$newborn$metrics,
      multinom_results$non_newborn$metrics
    )
  )
}

if (exists("rf_cls_results")) {
  classification_result_list <- c(
    classification_result_list,
    list(
      rf_cls_results$newborn$metrics,
      rf_cls_results$non_newborn$metrics
    )
  )
}

if (exists("cat_cls_results")) {
  classification_result_list <- c(
    classification_result_list,
    list(
      cat_cls_results$newborn$metrics,
      cat_cls_results$non_newborn$metrics
    )
  )
}

classification_summary <- bind_rows(classification_result_list) %>%
  mutate(across(where(is.numeric), ~ round(.x, 4)))

write.csv(
  classification_summary,
  file.path(output_path, "04_classification_model_summary.csv"),
  row.names = FALSE
)

classification_summary


Dataset,Model,Accuracy,Macro_F1,Status
<chr>,<chr>,<dbl>,<dbl>,<chr>
Newborn,Multinomial Logistic Regression,0.5962,0.4209,NA
Newborn,Random Forest Classifier - Target,0.5923,0.3867,NA
Non-newborn,Random Forest Classifier - Target,0.4530,0.4219,NA
Newborn,CatBoost Classifier,NA,NA,Not run: catboost package unavailable in current R environment
Non-newborn,CatBoost Classifier,NA,NA,Not run: catboost package unavailable in current R environment


## 4.8 Hyperparameter Setting

本節彙整原論文 Table 16 所列模型參數。Random Forest Regression 使用 10 estimators 與 maximum depth 10；Decision Tree Regression maximum depth 5。Logistic Regression 原論文以 Adam optimizer 設定 learning rate 1e-3、weight decay 1e-4、epochs 10；本 R 重現版本使用 `nnet::multinom()` 作為多類別 logistic regression 的 R 等價實作，因此不使用 Adam optimizer。

In [33]:
#--------------------------------------------------
# 4.8 Hyperparameter Setting Summary
#--------------------------------------------------

hyperparameter_summary <- tibble(
  Model = c(
    "Logistic Regression",
    "Logistic Regression",
    "Logistic Regression",
    "Random Forest Regression / Classifier",
    "Random Forest Regression / Classifier",
    "Decision Tree Regression"
  ),
  Parameter = c(
    "Adam optimizer learning rate",
    "Adam optimizer weight decay",
    "Adam optimizer number of epochs",
    "Number of estimators",
    "Maximum depth",
    "Maximum depth"
  ),
  Original_Value = c("1e-3", "1e-4", "10", "10", "10", "5"),
  R_Reproduction_Implementation = c(
    "nnet::multinom() R-equivalent; Adam not used",
    "nnet::multinom() R-equivalent; Adam not used",
    "nnet::multinom() R-equivalent; maxit = 100",
    "ranger::ranger(num.trees = 10)",
    "ranger::ranger(max.depth = 10)",
    "Reserved for classification tree visualization"
  )
)

write.csv(hyperparameter_summary, file.path(output_path, "04_hyperparameter_summary.csv"), row.names = FALSE)
hyperparameter_summary

Model,Parameter,Original_Value,R_Reproduction_Implementation
<chr>,<chr>,<chr>,<chr>
Logistic Regression,Adam optimizer learning rate,1e-3,nnet::multinom() R-equivalent; Adam not used
Logistic Regression,Adam optimizer weight decay,1e-4,nnet::multinom() R-equivalent; Adam not used
Logistic Regression,Adam optimizer number of epochs,10,nnet::multinom() R-equivalent; maxit = 200
Random Forest Regression / Classifier,Number of estimators,10,ranger::ranger(num.trees = 10)
Random Forest Regression / Classifier,Maximum depth,10,ranger::ranger(max.depth = 10)
Decision Tree Regression,Maximum depth,5,Reserved for classification tree visualization


## 4.9 Export Model Outputs for Evaluation

本節為 04_Model_Development 的 Freeze 版本新增輸出區塊。此區塊**不重新訓練模型、不修改模型參數、不改變任何預測結果**，僅將已完成的 metrics、predictions、confusion matrices 與可取得的 probabilities 匯出，供 05_Model_Evaluation.ipynb 直接讀取使用。

輸出目的：

- Regression：提供 Actual、Predicted、Residual，支援 R²、p-value、scatter plot、density plot 與 residual analysis。
- Classification：提供 Truth、Prediction、confusion matrix 與可取得的 class probabilities，支援 classification report、ROC / AUC 與 Brier score。
- CatBoost：若環境無法執行，保留 Not run 狀態，避免因套件限制改變研究設計。


In [ ]:
#--------------------------------------------------
# 4.9 Export Model Outputs for 05_Model_Evaluation
#--------------------------------------------------

regression_output_path <- file.path(output_path, "Regression")
classification_output_path <- file.path(output_path, "Classification")

dir.create(regression_output_path, recursive = TRUE, showWarnings = FALSE)
dir.create(classification_output_path, recursive = TRUE, showWarnings = FALSE)

#--------------------------------------------------
# 4.9.1 Regression metrics
#--------------------------------------------------

if (exists("regression_comparison_table")) {
  write.csv(
    regression_comparison_table,
    file.path(regression_output_path, "Regression_metrics.csv"),
    row.names = FALSE
  )
}

#--------------------------------------------------
# 4.9.2 Regression predictions
#--------------------------------------------------

make_regression_prediction_table <- function(actual, predicted, dataset_name, model_name) {
  tibble(
    Dataset = dataset_name,
    Model = model_name,
    Actual = as.numeric(actual),
    Predicted = as.numeric(predicted),
    Residual = as.numeric(actual) - as.numeric(predicted)
  )
}

regression_prediction_list <- list()

if (exists("linear_onehot_results")) {
  regression_prediction_list <- c(
    regression_prediction_list,
    list(
      make_regression_prediction_table(
        actual = nb_test_x$LOS,
        predicted = linear_onehot_results$newborn$prediction,
        dataset_name = "Newborn",
        model_name = "Linear Regression - One Hot"
      ),
      make_regression_prediction_table(
        actual = nnb_test_x$LOS,
        predicted = linear_onehot_results$non_newborn$prediction,
        dataset_name = "Non-newborn",
        model_name = "Linear Regression - One Hot"
      )
    )
  )
}

if (exists("rf_reg_results")) {
  regression_prediction_list <- c(
    regression_prediction_list,
    list(
      make_regression_prediction_table(
        actual = nb_test_x$LOS,
        predicted = rf_reg_results$newborn$prediction,
        dataset_name = "Newborn",
        model_name = "Random Forest Regression - Target"
      ),
      make_regression_prediction_table(
        actual = nnb_test_x$LOS,
        predicted = rf_reg_results$non_newborn$prediction,
        dataset_name = "Non-newborn",
        model_name = "Random Forest Regression - Target"
      )
    )
  )
}

regression_predictions <- bind_rows(regression_prediction_list)

write.csv(
  regression_predictions,
  file.path(regression_output_path, "Regression_predictions_all.csv"),
  row.names = FALSE
)

if (nrow(regression_predictions) > 0) {
  regression_prediction_files <- regression_predictions %>%
    distinct(Dataset) %>%
    pull(Dataset)

  for (dataset_i in regression_prediction_files) {
    file_stub <- dataset_i %>%
      str_replace_all("-", "_") %>%
      str_replace_all(" ", "_") %>%
      str_to_lower()

    write.csv(
      regression_predictions %>% filter(Dataset == dataset_i),
      file.path(regression_output_path, paste0(file_stub, "_regression_predictions.csv")),
      row.names = FALSE
    )
  }
}

#--------------------------------------------------
# 4.9.3 Classification metrics
#--------------------------------------------------

if (exists("classification_summary")) {
  write.csv(
    classification_summary,
    file.path(classification_output_path, "Classification_metrics.csv"),
    row.names = FALSE
  )
}

#--------------------------------------------------
# 4.9.4 Classification predictions
#--------------------------------------------------

make_classification_prediction_table <- function(result_obj, dataset_name, model_name) {
  if (is.null(result_obj$actual) || is.null(result_obj$prediction)) {
    return(tibble())
  }

  tibble(
    Dataset = dataset_name,
    Model = model_name,
    Truth = as.character(result_obj$actual),
    Prediction = as.character(result_obj$prediction)
  ) %>%
    filter(!is.na(Prediction))
}

classification_prediction_list <- list()

if (exists("multinom_results")) {
  classification_prediction_list <- c(
    classification_prediction_list,
    list(
      make_classification_prediction_table(
        multinom_results$newborn,
        "Newborn",
        "Multinomial Logistic Regression"
      ),
      make_classification_prediction_table(
        multinom_results$non_newborn,
        "Non-newborn",
        "Multinomial Logistic Regression"
      )
    )
  )
}

if (exists("rf_cls_results")) {
  classification_prediction_list <- c(
    classification_prediction_list,
    list(
      make_classification_prediction_table(
        rf_cls_results$newborn,
        "Newborn",
        "Random Forest Classifier - Target"
      ),
      make_classification_prediction_table(
        rf_cls_results$non_newborn,
        "Non-newborn",
        "Random Forest Classifier - Target"
      )
    )
  )
}

if (exists("cat_cls_results")) {
  classification_prediction_list <- c(
    classification_prediction_list,
    list(
      make_classification_prediction_table(
        cat_cls_results$newborn,
        "Newborn",
        "CatBoost Classifier"
      ),
      make_classification_prediction_table(
        cat_cls_results$non_newborn,
        "Non-newborn",
        "CatBoost Classifier"
      )
    )
  )
}

classification_predictions <- bind_rows(classification_prediction_list)

write.csv(
  classification_predictions,
  file.path(classification_output_path, "Classification_predictions_all.csv"),
  row.names = FALSE
)

if (nrow(classification_predictions) > 0) {
  classification_prediction_files <- classification_predictions %>%
    distinct(Dataset) %>%
    pull(Dataset)

  for (dataset_i in classification_prediction_files) {
    file_stub <- dataset_i %>%
      str_replace_all("-", "_") %>%
      str_replace_all(" ", "_") %>%
      str_to_lower()

    write.csv(
      classification_predictions %>% filter(Dataset == dataset_i),
      file.path(classification_output_path, paste0(file_stub, "_classification_predictions.csv")),
      row.names = FALSE
    )
  }
}

#--------------------------------------------------
# 4.9.5 Classification probabilities
#--------------------------------------------------

make_probability_table <- function(result_obj, dataset_name, model_name) {
  if (is.null(result_obj$actual) || is.null(result_obj$probability)) {
    return(tibble())
  }

  prob_df <- as.data.frame(result_obj$probability)

  # 統一欄位名稱，避免 1、2、3、4-6、>6 造成後續公式解析問題
  expected_names <- paste0("Prob_Class_", c("1", "2", "3", "4_6", "gt6"))

  if (ncol(prob_df) == length(expected_names)) {
    names(prob_df) <- expected_names
  } else {
    names(prob_df) <- paste0("Prob_Class_", seq_len(ncol(prob_df)))
  }

  bind_cols(
    tibble(
      Dataset = dataset_name,
      Model = model_name,
      Truth = as.character(result_obj$actual)
    ),
    prob_df
  )
}

classification_probability_list <- list()

if (exists("multinom_results")) {
  classification_probability_list <- c(
    classification_probability_list,
    list(
      make_probability_table(
        multinom_results$newborn,
        "Newborn",
        "Multinomial Logistic Regression"
      ),
      make_probability_table(
        multinom_results$non_newborn,
        "Non-newborn",
        "Multinomial Logistic Regression"
      )
    )
  )
}

if (exists("cat_cls_results")) {
  classification_probability_list <- c(
    classification_probability_list,
    list(
      make_probability_table(
        cat_cls_results$newborn,
        "Newborn",
        "CatBoost Classifier"
      ),
      make_probability_table(
        cat_cls_results$non_newborn,
        "Non-newborn",
        "CatBoost Classifier"
      )
    )
  )
}

classification_probabilities <- bind_rows(classification_probability_list)

if (nrow(classification_probabilities) > 0) {
  write.csv(
    classification_probabilities,
    file.path(classification_output_path, "Classification_probabilities_all.csv"),
    row.names = FALSE
  )
}

#--------------------------------------------------
# 4.9.6 Confusion matrices
#--------------------------------------------------

write_confusion_matrix <- function(confusion_obj, dataset_name, model_name) {
  if (is.null(confusion_obj)) {
    return(NULL)
  }

  file_stub <- paste(dataset_name, model_name, sep = "_") %>%
    str_replace_all("-", "_") %>%
    str_replace_all(" ", "_") %>%
    str_replace_all("[^A-Za-z0-9_]", "") %>%
    str_to_lower()

  confusion_df <- as.data.frame.matrix(confusion_obj) %>%
    tibble::rownames_to_column("Actual")

  write.csv(
    confusion_df,
    file.path(classification_output_path, paste0(file_stub, "_confusion_matrix.csv")),
    row.names = FALSE
  )

  return(file_stub)
}

confusion_exported <- c()

if (exists("multinom_results")) {
  confusion_exported <- c(
    confusion_exported,
    write_confusion_matrix(
      multinom_results$newborn$confusion_matrix,
      "Newborn",
      "Multinomial Logistic Regression"
    ),
    write_confusion_matrix(
      multinom_results$non_newborn$confusion_matrix,
      "Non-newborn",
      "Multinomial Logistic Regression"
    )
  )
}

if (exists("rf_cls_results")) {
  confusion_exported <- c(
    confusion_exported,
    write_confusion_matrix(
      rf_cls_results$newborn$confusion_matrix,
      "Newborn",
      "Random Forest Classifier - Target"
    ),
    write_confusion_matrix(
      rf_cls_results$non_newborn$confusion_matrix,
      "Non-newborn",
      "Random Forest Classifier - Target"
    )
  )
}

if (exists("cat_cls_results")) {
  confusion_exported <- c(
    confusion_exported,
    write_confusion_matrix(
      cat_cls_results$newborn$confusion_matrix,
      "Newborn",
      "CatBoost Classifier"
    ),
    write_confusion_matrix(
      cat_cls_results$non_newborn$confusion_matrix,
      "Non-newborn",
      "CatBoost Classifier"
    )
  )
}

confusion_exported <- confusion_exported[!is.na(confusion_exported)]

#--------------------------------------------------
# 4.9.7 Export log
#--------------------------------------------------

export_log <- tibble(
  Output = c(
    "Regression metrics",
    "Regression predictions",
    "Classification metrics",
    "Classification predictions",
    "Classification probabilities",
    "Confusion matrices"
  ),
  Status = c(
    ifelse(exists("regression_comparison_table"), "Exported", "Not available"),
    ifelse(exists("regression_predictions") && nrow(regression_predictions) > 0, "Exported", "Not available"),
    ifelse(exists("classification_summary"), "Exported", "Not available"),
    ifelse(exists("classification_predictions") && nrow(classification_predictions) > 0, "Exported", "Not available"),
    ifelse(exists("classification_probabilities") && nrow(classification_probabilities) > 0, "Exported", "Not available"),
    ifelse(length(confusion_exported) > 0, "Exported", "Not available")
  ),
  Folder = c(
    regression_output_path,
    regression_output_path,
    classification_output_path,
    classification_output_path,
    classification_output_path,
    classification_output_path
  )
)

write.csv(
  export_log,
  file.path(output_path, "04_export_log.csv"),
  row.names = FALSE
)

export_log


## 4.10 Training Pipeline Summary

本章完成原論文 Methods 中模型開發流程之 Reproduction，包括 target variable 建立、90/10 holdout split、encoding strategy、regression models、classification models 與 model parameter summary。

下一章建議進入 05_Model_Evaluation.ipynb，集中整理 regression R²/p-value、classification accuracy/F1、confusion matrix、ROC/AUC、Brier score 與 DeLong test 等模型效能評估結果。

本章已於 4.9 匯出 05_Model_Evaluation 所需之模型評估資料；因此 04 到此封版，後續章節原則上不再重新訓練模型。

In [34]:
#--------------------------------------------------
# 4.10 Training Pipeline Summary
#--------------------------------------------------

pipeline_summary <- tibble(
  Step = c(
    "Target variable",
    "Regression target",
    "Classification target",
    "Data split",
    "Cross-validation",
    "Encoding",
    "Regression models",
    "Classification models",
    "Export model outputs",
    "Not included in reproduction"
  ),
  Description = c(
    "Length of Stay (LoS)",
    "Continuous LoS from 1 to 120 days",
    "1, 2, 3, 4–6, >6 days",
    "90% training / 10% holdout test",
    "Tenfold cross-validation within training phase",
    "One-hot, target encoding, and combined encoding concept",
    "Linear Regression, Random Forest Regression, CatBoost Regression",
    "Multinomial Logistic Regression, Random Forest Classifier, CatBoost Classifier",
    "Metrics, predictions, probabilities if available, and confusion matrices for 05_Model_Evaluation",
    "XGBoost and LightGBM, because they were not used in Jain et al. (2024)"
  )
)

write.csv(pipeline_summary, file.path(output_path, "04_training_pipeline_summary.csv"), row.names = FALSE)
pipeline_summary

Step,Description
<chr>,<chr>
Target variable,Length of Stay (LoS)
Regression target,Continuous LoS from 1 to 120 days
Classification target,"1, 2, 3, 4–6, >6 days"
Data split,90% training / 10% holdout test
Cross-validation,Tenfold cross-validation within training phase
Encoding,"One-hot, target encoding, and combined encoding concept"
Regression models,"Linear Regression, Random Forest Regression, CatBoost Regression"
Classification models,"Multinomial Logistic Regression, Random Forest Classifier, CatBoost Classifier"
Not included in reproduction,"XGBoost and LightGBM, because they were not used in Jain et al. (2024)"
